# 跑yolo模型

## 第一步：环境配置

**1.GPU检查**

代码：！是执行系统shell命令前缀

结果：

分配T4 GPU，检查GPU是否分配成功

Perf表示性能状态，P0最高，p12最低。

memory-usage:显存使用情况

GPU-util:GPU利用率

In [1]:
!nvidia-smi

Wed Jun  3 12:51:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**2.挂云盘**：这是colab专门提供和google drive交互的模块

mount是挂载的意思，挂载到xx文件夹  

打开左侧文件可以看到，colab上的文件，关机后会消失，而drive/mydrive文件夹的内容
是保存到自己电脑云端的

colab本地运行的更快，需要频繁读写的文件如克隆的仓库和数据集都要放到colab  
放到drive的文件：训练好的.pt，导出的历史轨迹.csv等等

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


**3.安装所有必要的库**

yolo的库

获取数据集文件的库

卡尔曼滤波的库

In [3]:
!pip install ultralytics roboflow filterpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 7.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 145.3 MB/s eta 0:00:00
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=d84cde0a14e8504888d069a43ab6ed22358dd0e0dd410d17eb675376fff77e68
  Stored in directory: /root/.cache/pip/wheels/77/bf/4c/b0c3f4798a0166668752312a67118b27a3cd341e13ac0ae6ee
Successfully built filterpy
  Attempting uninstall: opencv-python-

**4.创建目录**

将所有的代码和配置都放在 /content/autonomous_driving 目录下

In [4]:
import os

PROJECT_PATH = "/content/autonomous_driving"
for folder in ["configs", "src", "weights", "data", "results"]:
    os.makedirs(os.path.join(PROJECT_PATH, folder), exist_ok=True)

print(f"✅ 项目目录 {PROJECT_PATH} 已建立！")

✅ 项目目录 /content/autonomous_driving 已建立！


## 第二步：获取并处理数据集

**5.获取数据集**

出现的问题：本地下载的数据集文件在google云盘下载的太慢且网络波动后断掉了。

解决方案：用别人提供的文件。

找了个Roboflow平台（计算机视觉数据管理预处理）

在其中找到kitti数据集下载下来

In [7]:
%cd /content/autonomous_driving/data

from roboflow import Roboflow
rf = Roboflow(api_key="nejPKc5WYocno3KDVt6j")
project = rf.workspace("sudip-dhakal").project("kitti-uajb1")
version = project.version(1)
dataset = version.download("yolov8")



#  检查并重命名
# 下载后的文件夹通常叫 "KITTI-1"，我们把它改名为 "kitti_raw"
import os
if os.path.exists("/content/autonomous_driving/data/KITTI-1"):
    os.rename("/content/autonomous_driving/data/KITTI-1", "/content/autonomous_driving/data/kitti_raw")
    print("✅ 数据集已成功下载并重命名为 kitti_raw")
else:
    # 有时候名字里带空格或版本号，如果改名失败，我们打印一下当前目录看看它叫什么
    print("当前目录下的文件夹有：", os.listdir())

/content/autonomous_driving/data
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to KITTI-1 in yolov8:: 100%|██████████| 14968/14968 [00:03<00:00, 4525.91it/s]


✅ 数据集已成功下载并重命名为 kitti_raw


In [8]:
# 看看训练集里到底有多少张图片
!ls /content/autonomous_driving/data/kitti_raw/train/images | wc -l

7481


**6.划分数据集与验证集**

In [9]:
import random
import shutil

train_img_path = "/content/autonomous_driving/data/kitti_raw/train/images"
train_lab_path = "/content/autonomous_driving/data/kitti_raw/train/labels"
valid_img_path = "/content/autonomous_driving/data/kitti_raw/valid/images"
valid_lab_path = "/content/autonomous_driving/data/kitti_raw/valid/labels"

os.makedirs(valid_img_path, exist_ok=True)
os.makedirs(valid_lab_path, exist_ok=True)

all_images = [f for f in os.listdir(train_img_path) if f.endswith('.jpg')]
val_count = int(len(all_images) * 0.1)
#列表随机抽取
val_images = random.sample(all_images, val_count)

for img_name in val_images:
    #移动图片
    shutil.move(os.path.join(train_img_path, img_name), os.path.join(valid_img_path, img_name))
    lab_name = img_name.replace('.jpg', '.txt')
    if os.path.exists(os.path.join(train_lab_path, lab_name)):
        shutil.move(os.path.join(train_lab_path, lab_name), os.path.join(valid_lab_path, lab_name))

print(f"✅ 数据集重新划分完成，验证集现在有 {len(val_images)} 张图。")

✅ 数据集重新划分完成，验证集现在有 748 张图。


In [10]:
# 统计训练集图片数量 (预期应该是 7481 - 748 = 6733 左右)
print("训练集图片数：")
!ls /content/autonomous_driving/data/kitti_raw/train/images | wc -l

# 统计验证集图片数量 (预期应该是 748)
print("验证集图片数：")
!ls /content/autonomous_driving/data/kitti_raw/valid/images | wc -l

训练集图片数：
6733
验证集图片数：
748


## 第三步：训练

**7.生成yaml文件**

In [12]:
import yaml

data_config = {
    'path': '/content/autonomous_driving/data/kitti_raw', # 绝对路径
    'train': 'train/images',
    'val': 'valid/images',
    'nc': 2,
    'names': ['car', 'person']
}

with open('/content/autonomous_driving/configs/kitti_config.yaml', 'w') as f:
    yaml.dump(data_config, f)

print("✅ 配置文件已生成：/content/autonomous_driving/configs/kitti_config.yaml")

✅ 配置文件已生成：/content/autonomous_driving/configs/kitti_config.yaml


**8.正式训练**

In [ ]:
from ultralytics import YOLO

# 加载模型
model = YOLO("yolov8s.pt")

# 启动训练
results = model.train(
    data="/content/autonomous_driving/configs/kitti_config.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=8,
    project="/content/autonomous_driving/results",
    name="kitti_v8s_full"
)

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/autonomous_driving/configs/kitti_config.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=kitti_v8s_full, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, ov

**9.保存下来**

In [ ]:
import shutil
import os

# 定义云盘备份路径
backup_dir = "/content/drive/MyDrive/Lantu_Project_Backup"
os.makedirs(backup_dir, exist_ok=True)

# 1. 备份最好的权重
shutil.copy("/content/autonomous_driving/results/kitti_v8s_full/weights/best.pt", f"{backup_dir}/best.pt")

# 2. 备份训练指标图
shutil.copy("/content/autonomous_driving/results/kitti_v8s_full/results.png", f"{backup_dir}/results.png")
shutil.copy("/content/autonomous_driving/results/kitti_v8s_full/confusion_matrix.png", f"{backup_dir}/confusion_matrix.png")

print(f"✅ 所有的‘宝贝’都已安全备份到云盘：{backup_dir}")

✅ 所有的‘宝贝’都已安全备份到云盘：/content/drive/MyDrive/Lantu_Project_Backup


# 感知


## 环境与装载best.pt

1.加载模型并验证map

In [13]:
from ultralytics import YOLO

# 确认你的 best.pt 路径（就按你 notebook Cell 27 里那个路径）
model = YOLO("/content/drive/MyDrive/YOLO_Project/best.pt")

# 验证集评估
metrics = model.val(
    data="/content/autonomous_driving/configs/kitti_config.yaml",
    imgsz=640
)
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")


Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,126,358 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2103.3±1024.8 MB/s, size: 84.5 KB)
val: Scanning /content/autonomous_driving/data/kitti_raw/valid/labels... 748 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 748/748 2.1Kit/s 0.4s
val: New cache created: /content/autonomous_driving/data/kitti_raw/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 6.4it/s 7.3s
                   all        748       3190       0.94      0.855      0.919       0.71
                   car        664       2787      0.952      0.944      0.981      0.838
                person        178        403      0.928      0.766      0.858      0.581
Speed: 0.4ms preprocess, 3.1ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to /content/autonomous_

2.图片序列跟踪

In [14]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/YOLO_Project/best.pt")

results = model.track(
    source="/content/autonomous_driving/data/kitti_raw/valid/images",
    tracker="botsort.yaml",
    conf=0.4,
    persist=True,
    save=True,
    project="/content/autonomous_driving/results",
    name="tracking_demo"
)
print("完成！左侧文件栏 → autonomous_driving/results/tracking_demo/ 看结果图片")


requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 348ms
Prepared 1 package in 80ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


image 1/748 /content/autonomous_driving/data/kitti_raw/valid/images/000002_png.rf.5cb81dbce207ee7fb95e2a5a425cc982.jpg: 224x640 1 car, 74.9ms
image 2/748 /content/autonomous_driving/data/kitti_raw/valid/images/000008_png.rf.a72d425910a10f7946321fe5b668fec4.jpg: 224x640 1 car, 9.3ms
image 3/748 /content/autonomous_driving/data/kitti_raw/valid/images/000012_png.rf.4eee3142489e445e4ce8d1fe397767e7.jpg: 224x640 1 car, 10.9ms
image 4/748 /content/autonomous_driving/data/kitti_raw/valid/images/000037_png.rf.b97356e464f7fbaed1e14bff9c9dccf2.jpg: 224x640 3 cars, 10.4ms
image 5/748 /content/autonomous_driving/data/kitti_raw/valid/images/000057

3.备份

In [15]:
import shutil, os

backup = "/content/drive/MyDrive/Lantu_Project_Backup"
os.makedirs(backup, exist_ok=True)
shutil.copytree(
    "/content/autonomous_driving/results/tracking_demo",
    f"{backup}/tracking_demo",
    dirs_exist_ok=True
)
print("✅ 备份完成")


✅ 备份完成
